# Pixelated Empathy - Stage 1 Training

**One-click training notebook for Stage 1 foundation training**

---

### Quick Start
1. **Runtime -> Change runtime type -> GPU -> A100** (or V100/T4)
2. **Run all cells** (Ctrl+F9 or Runtime -> Run all)
3. **Done!** Training will start automatically

---

### Requirements
- GPU Runtime (A100 recommended, V100/T4 works)
- ~30GB VRAM for full training
- Internet connection for S3 data download

---

### Expected Runtime
- **A100 (40GB)**: ~2-4 hours
- **V100 (16GB)**: ~4-8 hours
- **T4 (16GB)**: ~8-16 hours (lower batch size)

In [ ]:
# @title Step 1: Set Environment Variables
import logging
import os

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("stage1-training")

_required_secrets = (
    "OVH_S3_ACCESS_KEY",
    "OVH_S3_SECRET_KEY",
    "WANDB_API_KEY",
    "HF_TOKEN",
)

def _load_secret_from_colab(name: str):
    if os.environ.get(name):
        return os.environ[name]

    try:
        from google.colab import userdata
    except Exception:
        return None

    try:
        value = userdata.get(name)
    except Exception:
        return None

    return value

for key in _required_secrets:
    value = _load_secret_from_colab(key)
    if value:
        os.environ[key] = value

# Non-sensitive defaults for local paths/project metadata.
os.environ.setdefault("OVH_S3_BUCKET", "pixel-data")
os.environ.setdefault("OVH_S3_ENDPOINT", "https://s3.us-east-va.io.cloud.ovh.us")
os.environ.setdefault("OVH_S3_REGION", "us-east-va")
os.environ.setdefault("WANDB_PROJECT", "pixelated-empathy")
os.environ.setdefault("HF_PROJECT_NAME", "pixelated-empathy-stage1")
os.environ.setdefault("PIXELATED_DRIVE_DIR", "/content/drive/MyDrive/pixelated-checkpoints")

missing_required_secrets = [key for key in _required_secrets if not os.environ.get(key)]
if missing_required_secrets:
    logger.warning("Missing required environment variables:")
    for key in missing_required_secrets:
        logger.warning(f"   - {key}")
else:
    logger.info("All required environment variables are set.")

os.environ["STAGE1_REQUIRE_PREFLIGHT"] = os.environ.get("STAGE1_REQUIRE_PREFLIGHT", "1")

logger.info("Environment defaults configured.")
logger.info(f"   S3 Bucket: {os.environ['OVH_S3_BUCKET']}")
logger.info(f"   W&B Project: {os.environ['WANDB_PROJECT']}")
logger.info(f"   HF Project: {os.environ['HF_PROJECT_NAME']}")


Environment variables set!
   S3 Bucket: pixel-data
   W&B Project: pixelated-empathy
   HF Token: **********...jaoe


In [ ]:
# @title Step 2: Verify GPU
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True, check=True
)

if result.returncode == 0:
    print("GPU Detected:")
    for line in result.stdout.strip().split('\n'):
        print(f"   {line}")
else:
    print("No GPU detected! Please check:")
    print("   1. Runtime -> Change runtime type -> GPU")
    print("   2. Make sure GPU is selected (A100/V100/T4)")
    raise SystemExit("No GPU available")

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

GPU Detected:
   NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB, 97251 MiB

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
BF16 supported: True


In [ ]:
# @title Step 3: Install Dependencies
import subprocess
import sys

print("Installing packages...")

packages = [
    "unsloth",
    "transformers",
    "datasets",
    "peft",
    "accelerate",
    "bitsandbytes",
    "trl",
    "boto3",
    "wandb",
    "huggingface_hub",
]

for pkg in packages:
    print(f"   Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\nAll dependencies installed!")

Installing packages...
   Installing unsloth...
   Installing transformers...
   Installing datasets...
   Installing peft...
   Installing accelerate...
   Installing bitsandbytes...
   Installing trl...
   Installing boto3...
   Installing wandb...
   Installing huggingface_hub...

All dependencies installed!


In [ ]:
# @title Step 4: Download Training Data from S3 (Sharded Format)
import os
from pathlib import Path
import boto3
from botocore.config import Config

data_dir = Path("/content/data")
data_dir.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {data_dir}")

s3_bucket = os.environ.get("OVH_S3_BUCKET", "pixel-data")
s3_endpoint = os.environ.get("OVH_S3_ENDPOINT", "https://s3.us-east-va.io.cloud.ovh.us")
access_key = os.environ.get("OVH_S3_ACCESS_KEY")
secret_key = os.environ.get("OVH_S3_SECRET_KEY")

print(f"S3 Bucket: {s3_bucket}")
print(f"S3 Endpoint: {s3_endpoint}")

s3 = boto3.client(
    "s3",
    endpoint_url=s3_endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    config=Config(signature_version="s3v4"),
)

# Download sharded files - train_shard_XXX.jsonl and val_shard_XXX.jsonl
print("\n=== Downloading Training Data ===")

train_files = []
val_files = []

# List all shards
result = s3.list_objects_v2(Bucket=s3_bucket, Prefix="compiled_dataset/train_shard_")
if "Contents" in result:
    train_files = sorted([obj["Key"] for obj in result['Contents'] if obj["Key"].endswith('.jsonl')])
    print(f"Found {len(train_files)} training shards")

result = s3.list_objects_v2(Bucket=s3_bucket, Prefix="compiled_dataset/val_shard_")
if "Contents" in result:
    val_files = sorted([obj["Key"] for obj in result['Contents'] if obj["Key"].endswith('.jsonl')])
    print(f"Found {len(val_files)} validation shards")

if not train_files:
    print("ERROR: No training files found on S3!")
    raise SystemExit("S3 data not found")

# Download first N training shards and all validation shards
# Adjust these numbers based on available time/VRAM
MAX_TRAIN_SHARDS = 50  # Download first 50 shards (~1GB)
MAX_VAL_SHARDS = 5     # Download first 5 validation shards

train_output = data_dir / "train.jsonl"
val_output = data_dir / "val.jsonl"

# Combine training shards
print(f"\nDownloading {min(MAX_TRAIN_SHARDS, len(train_files))} training shards...")
with open(train_output, "w") as out_f:
    for i, key in enumerate(train_files[:MAX_TRAIN_SHARDS]):
        print(f"   [{i+1}/{min(MAX_TRAIN_SHARDS, len(train_files))}] {key}")
        obj = s3.get_object(Bucket=s3_bucket, Key=key)
        out_f.write(obj["Body"].read().decode("utf-8"))
        if i < MAX_TRAIN_SHARDS - 1 and i < len(train_files) - 1:
            out_f.write('\n')  # Add newline between shards

print(f"   Combined into: {train_output} ({train_output.stat().st_size / 1024 / 1024:.1f} MB)")

# Combine validation shards
print(f"\nDownloading {min(MAX_VAL_SHARDS, len(val_files))} validation shards...")
with open(val_output, "w") as out_f:
    for i, key in enumerate(val_files[:MAX_VAL_SHARDS]):
        print(f"   [{i+1}/{min(MAX_VAL_SHARDS, len(val_files))}] {key}")
        obj = s3.get_object(Bucket=s3_bucket, Key=key)
        out_f.write(obj["Body"].read().decode("utf-8"))
        if i < MAX_VAL_SHARDS - 1 and i < len(val_files) - 1:
            out_f.write('\n')

print(f"   Combined into: {val_output} ({val_output.stat().st_size / 1024 / 1024:.1f} MB)")

print("\nData download complete!")

Data directory: /content/data
S3 Bucket: pixel-data
S3 Endpoint: https://s3.us-east-va.io.cloud.ovh.us

=== Downloading Training Data ===
Found 139 training shards
Found 7 validation shards

   [1/50] compiled_dataset/train_shard_000.jsonl
   [2/50] compiled_dataset/train_shard_001.jsonl
   [3/50] compiled_dataset/train_shard_002.jsonl
   [4/50] compiled_dataset/train_shard_003.jsonl
   [5/50] compiled_dataset/train_shard_004.jsonl
   [6/50] compiled_dataset/train_shard_005.jsonl
   [7/50] compiled_dataset/train_shard_006.jsonl
   [8/50] compiled_dataset/train_shard_007.jsonl
   [9/50] compiled_dataset/train_shard_008.jsonl
   [10/50] compiled_dataset/train_shard_009.jsonl
   [11/50] compiled_dataset/train_shard_010.jsonl
   [12/50] compiled_dataset/train_shard_011.jsonl
   [13/50] compiled_dataset/train_shard_012.jsonl
   [14/50] compiled_dataset/train_shard_013.jsonl
   [15/50] compiled_dataset/train_shard_014.jsonl
   [16/50] compiled_dataset/train_shard_015.jsonl
   [17/50] compile

In [ ]:
# @title Step 5: Initialize W&B
import os

wandb_api_key = os.environ.get('WANDB_API_KEY')

if wandb_api_key:
    import wandb
    wandb.login(key=wandb_api_key)

    project = os.environ.get('WANDB_PROJECT', 'pixelated-empathy')
    wandb.init(project=project, name="stage1-training")
    print(f"W&B initialized! View at: https://wandb.ai/{project}")
else:
    print("W&B not configured - training will proceed without tracking")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: wutang to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B initialized! View at: https://wandb.ai/pixelated-empathy


In [ ]:
# @title Step 6: Run Stage 1 Training
import os
import sys
from pathlib import Path
import torch
# Add the installation of unsloth here to ensure it's available
%pip install -q unsloth
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from datasets import load_dataset
from trl import SFTTrainer

print("=" * 60)
print("STAGE 1 TRAINING - Pixelated Empathy")
print("=" * 60)

DATA_DIR = Path("/content/data")
CHECKPOINT_DIR = Path("/content/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "LatitudeGames/Wayfarer-2-12B"
MAX_SEQ_LENGTH = 4096
LORA_R = 16

print(f"Data directory: {DATA_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Model: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

# Load model with Unsloth
print("\nLoading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    load_in_4bit=True,
    # Use fix_tokenizer to address tokenizer regex pattern warning
    fix_tokenizer=True,
)
print("Model loaded!")

# Add LoRA adapters
print("Adding LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("LoRA adapters configured!")

# Load dataset
train_file = DATA_DIR / "train.jsonl"
val_file = DATA_DIR / "val.jsonl"

print(f"\nLoading dataset from {train_file}...")
dataset = load_dataset("json", data_files=str(train_file), split="train")
print(f"Loaded {len(dataset)} training samples")

if val_file.exists():
    val_dataset = load_dataset("json", data_files=str(val_file), split="train")
    print(f"Loaded {len(val_dataset)} validation samples")
else:
    val_dataset = None

# Define a formatting function for SFTTrainer


def _select_field_key(examples, candidates):
    for key in candidates:
        if key in examples:
            return key
    return None


def _build_prompt_text(raw_prompt):
    if isinstance(raw_prompt, list):
        user_messages = [
            msg.get("content") for msg in raw_prompt
            if isinstance(msg, dict)
            and msg.get("role") == "user"
            and msg.get("content") is not None
        ]
        if user_messages:
            return user_messages[-1]

        text_messages = [msg for msg in raw_prompt if isinstance(msg, str)]
        if text_messages:
            return text_messages[-1]

        return ""

    if raw_prompt is None:
        return ""

    if isinstance(raw_prompt, dict) and isinstance(raw_prompt.get("content"), str):
        return raw_prompt["content"]

    return str(raw_prompt)


def _build_formatted_text(prompt_value, completion_value):
    prompt_text = _build_prompt_text(prompt_value)
    completion_text = str(completion_value)
    if completion_text.startswith("{") and completion_text.endswith("}"):
        completion_text = "[metadata json block]"

    return f"""### Instruction:
{prompt_text}

### Output:
{completion_text}"""


def _to_record_list(values):
    if values is None:
        return [""]

    if isinstance(values, (list, tuple)):
        return list(values)

    if isinstance(values, dict):
        if "content" in values and isinstance(values["content"], str):
            return [values["content"]]
        return [str(values)]

    return [values]


def _iter_examples_for_formatting(examples):
    if isinstance(examples, dict):
        list_values = [value for value in examples.values() if isinstance(value, (list, tuple))]
        if not list_values:
            yield examples
            return

        max_records = max(len(value) for value in list_values)
        for index in range(max_records):
            row = {}
            for key, value in examples.items():
                if isinstance(value, (list, tuple)):
                    row[key] = value[index] if index < len(value) else ""
                else:
                    row[key] = value
            yield row
        return

    if isinstance(examples, (list, tuple)):
        for item in examples:
            if isinstance(item, dict):
                yield item
        return

    return


def formatting_prompts_func(examples):
    prompt_candidates = ("messages", "prompt", "instruction")
    completion_candidates = ("metadata", "response", "completion", "output", "text")

    examples_list = list(_iter_examples_for_formatting(examples))
    if not examples_list:
        print("Error: formatting_prompts_func received empty or invalid examples")
        return []

    texts = []
    for example in examples_list:
        prompt_key = _select_field_key(example, prompt_candidates)
        completion_key = _select_field_key(example, completion_candidates)

        if prompt_key is None:
            print(f"Error: Dataset is missing a prompt key. Available keys: {list(example.keys())}")
            return []

        if completion_key is None:
            print(f"Error: Dataset is missing a completion key. Available keys: {list(example.keys())}")
            return []

        prompt_records = _to_record_list(example[prompt_key])
        completion_records = _to_record_list(example[completion_key])
        max_records = max(len(prompt_records), len(completion_records), 1)

        for index in range(max_records):
            prompt_value = prompt_records[index] if index < len(prompt_records) else ""
            completion_value = completion_records[index] if index < len(completion_records) else ""
            texts.append(_build_formatted_text(prompt_value, completion_value))

    return texts

# Training arguments - adjust based on GPU
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"\nOptimizing for: {gpu_name}")

if "A100" in gpu_name:
    per_device_batch_size = 2
    gradient_accumulation_steps = 8
elif "V100" in gpu_name:
    per_device_batch_size = 1
    gradient_accumulation_steps = 16
else:
    per_device_batch_size = 1
    gradient_accumulation_steps = 16

print(f"   Batch size: {per_device_batch_size}")
print(f"   Gradient accumulation: {gradient_accumulation_steps}")

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=per_device_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="wandb" if os.environ.get('WANDB_API_KEY') else "none",
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

print("\nInitializing trainer...")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text", # This now refers to the 'text' field created by formatting_prompts_func
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    formatting_func=formatting_prompts_func,
)

print("\n" + "=" * 60)
print("STARTING TRAINING...")
print("=" * 60)

trainer.train()

print("\nSaving model...")
model.save_pretrained(str(CHECKPOINT_DIR / "final_model"))
tokenizer.save_pretrained(str(CHECKPOINT_DIR / "final_model"))

print("\n" + "=" * 60)
print("STAGE 1 TRAINING COMPLETE!")
print("=" * 60)
print(f"Model saved to: {CHECKPOINT_DIR / 'final_model'}")
if os.environ.get('WANDB_API_KEY'):
    print(f"View training at: https://wandb.ai/{os.environ.get('WANDB_PROJECT', 'pixelated-empathy')}")

STAGE 1 TRAINING - Pixelated Empathy
Data directory: /content/data
Checkpoint directory: /content/checkpoints
Model: LatitudeGames/Wayfarer-2-12B
Max sequence length: 4096

Loading model...
==((====))==  Unsloth 2026.3.4: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

The tokenizer you are loading from 'LatitudeGames/Wayfarer-2-12B' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'LatitudeGames/Wayfarer-2-12B' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load LatitudeGames/Wayfarer-2-12B as a legacy tokenizer.


Model loaded!
Adding LoRA adapters...
LoRA adapters configured!

Loading dataset from /content/data/train.jsonl...
Loaded 250000 training samples
Loaded 25000 validation samples

Optimizing for: NVIDIA RTX PRO 6000 Blackwell Server Edition
   Batch size: 1
   Gradient accumulation: 16

Initializing trainer...


KeyError: 0

In [ ]:
import json

file_path = "/content/data/train.jsonl"

print(f"Inspecting first 5 lines of {file_path}:")
try:
    with open(file_path) as f:
        for i, line in enumerate(f):
            if i >= 5: # Limit to first 5 lines
                break
            try:
                data = json.loads(line.strip())
                print(f"Line {i+1}: {data.keys()}")
                print(f"   Sample line content: {json.dumps(data, indent=2)}")
            except json.JSONDecodeError:
                print(f"Line {i+1}: Invalid JSON")
except FileNotFoundError:
    print(f"Error: {file_path} not found. Please ensure data download completed successfully.")

In [ ]:
import os

file_path = "/content/data/train.jsonl"
if os.path.exists(file_path):
    print(f"The file {file_path} exists.")
else:
    print(f"The file {file_path} does NOT exist.")

In [ ]:
# @title Step 7: Upload to Hugging Face Hub
import os

hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    from huggingface_hub import HfApi, create_repo

    # Get username from token
    from huggingface_hub import HfLogin
    login = HfLogin(token=hf_token)
    api = HfApi()
    whoami = api.whoami()
    username = whoami['name']

    checkpoint_dir = "/content/checkpoints/final_model"
    repo_name = f"{username}/pixelated-empathy-stage1"

    print("Uploading to Hugging Face Hub...")
    print(f"   Repo: {repo_name}")

    try:
        create_repo(repo_name, token=hf_token, exist_ok=True)
    except Exception as e:
        print(f"   Repo may already exist: {e}")

    api.upload_folder(
        folder_path=checkpoint_dir,
        repo_id=repo_name,
        repo_type="model",
        token=hf_token,
    )

    print(f"\nUploaded to: https://huggingface.co/{repo_name}")
else:
    print("HF_TOKEN not set - skipping upload")

In [ ]:
# @title Step 8: Save Checkpoints to Google Drive (Optional)
try:
    from google.colab import drive
    import shutil
    from pathlib import Path

    print("Saving checkpoints to Google Drive...")
    drive.mount('/content/drive')

    checkpoint_dir = Path("/content/checkpoints")
    drive_dir = Path("/content/drive/MyDrive/pixelated-checkpoints")
    drive_dir.mkdir(parents=True, exist_ok=True)

    final_model = checkpoint_dir / "final_model"
    if final_model.exists():
        dest = drive_dir / "final_model"
        print(f"   Copying to {dest}...")
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(final_model, dest)
        print("   Saved to Drive!")
    else:
        print("   No final model found")

except ImportError:
    print("Google Drive not available - skipping")
except Exception as e:
    print(f"Could not save to Drive: {e}")

print("\nAll done!")

---

## Training Summary

| Stage | Stage 1 (Foundation Training) |
| Model | LatitudeGames/Wayfarer-2-12B |
| LoRA Rank | 16 |
| Max Sequence Length | 4096 |
| Epochs | 1 |
| Learning Rate | 2e-4 |

---

*Notebook generated for Pixelated Empathy - Stage 1 Training*